In [1]:
from dataclasses import dataclass
from torch import Tensor
from typing import Set, List, Optional

@dataclass
class movie:
    """
    Data class representing a movie with its metadata.
    """
    ryear: float          # Release year
    title: str            # Movie title
    runtime: int          # Runtime in minutes
    genres: Set[str]      # Set of genres
    overview: str         # Plot overview
    rating: float         # IMDB rating
    meta_score: Optional[float] # Metascore (can be None)
    directors: Set[str]   # Set of directors
    stars: Set[str]       # Set of main stars
    gross: Optional[float] # Gross revenue (can be None)

In [2]:
#load SBERT


from sentence_transformers import SentenceTransformer,util

sbert = SentenceTransformer('all-MiniLM-L6-v2')# load the pretrained model.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
import csv

#load movies from the input file
def load_movies(input_file:str):
    """
    Parses a CSV file containing movie data, creates movie objects,
    and computes embeddings for movie overviews.

    Args:
        input_file (str): Path to the CSV file.

    Returns:
        tuple: A tuple containing:
            - movies (list): A list of 'movie' objects.
            - sim_matrix (Tensor): A cosine similarity matrix of overview embeddings.
            - title_index (dict): A dictionary mapping movie titles to their index in the movies list.
    """

    title_index={}# title to index
    movies=[]
    with open(input_file) as f:
        overviews=[]

        for row in csv.DictReader(f): # for each movie

            #make a new movie object
            new_mv=movie(int(row['Released_Year']) if row['Released_Year'].isnumeric() else None,
                         row['Series_Title'],
                         int(row['Runtime'][:row['Runtime'].find(' ')]),
                         set([x.strip() for x in row['Genre'].split(',')]),
                         row['Overview'],
                         float(row['IMDB_Rating']),
                         float(row['Meta_score'][:row['Meta_score'].find(' ')]) if row['Meta_score']!='' else None,
                         set([x.strip() for x in row['Director'].split(',')]),
                         set([row[x] for x in ['Star1','Star2','Star3','Star4']]),
                         float(row['Gross'].replace(',','')) if row['Gross']!='' else None)

            #store the overview separately
            overviews.append(row['Overview'])

            #update the index
            title_index[row['Series_Title']]=len(movies)
            movies.append(new_mv)

    #encode all overviews
    embedded=sbert.encode(overviews,convert_to_tensor=True)

    #Compute cosine-similarities
    sim_matrix = util.cos_sim(embedded, embedded)

    return movies,sim_matrix,title_index

In [4]:
movies,overview_sim_matrix,title_index=load_movies('imdb_top_1000.csv')

lst=[m.ryear for m in movies if m.ryear!=None]

srt=sorted(lst)

mx=srt[-1]
mn=srt[0]

mn,mx

(1920, 2020)

In [5]:
overview_sim_matrix[1,10]

tensor(0.0851)

In [6]:
import numpy as np

#movie-to-movie similarity
def sim(title1:str,
           title2:str,
           movies:list,
           weights:dict,
           overview_sim_matrix:Tensor,
           title_index:dict)->float:
    """
    Computes a weighted similarity score between two movies based on various factors.

    Factors considered:
    - Stars (Jaccard similarity)
    - Directors (Jaccard similarity)
    - Genres (Jaccard similarity)
    - Release Year (Normalized difference)
    - Overview (Cosine similarity of SBERT embeddings)
    - Rating (Normalized IMDB rating of the candidate movie)

    Args:
        title1 (str): Title of the first movie (candidate).
        title2 (str): Title of the second movie (seed/reference).
        movies (list): List of movie objects.
        weights (dict): Dictionary of weights for each similarity factor.
        overview_sim_matrix (Tensor): Precomputed similarity matrix for overviews.
        title_index (dict): Lookup dictionary for movie indices.

    Returns:
        tuple: (overall_score, explanations)
            - overall_score (float): The weighted sum of similarity scores.
            - explanations (list): A sorted list of tuples (factor, weighted_score) explaining the score.
    """

    mid1,mid2=title_index[title1],title_index[title2]# get the movie ids (idnexes)
    m1=movies[mid1] # get the movie objects
    m2=movies[mid2]

    scores=dict() # stores a the score for each factor from the weights dict

    #star jacard
    scores['star']=len(m1.stars.intersection(m2.stars))/len(m1.stars.union(m2.stars))

    #director jaccard
    scores['director']=len(m1.directors.intersection(m2.directors))/len(m1.directors.union(m2.directors))

    #genre jaccard
    scores['genre']=len(m1.genres.intersection(m2.genres))/len(m1.genres.union(m2.genres))

    # release year diff
    try:
        scores['ryear']=1- abs(m1.ryear-m2.ryear)/(mx-mn)
    except:
        scores['ryear']=0

    #cosine sim for overviews
    scores['overview']=overview_sim_matrix[mid1,mid2].numpy()

    #normalized candidate rating
    scores['rating']=m1.rating/10

    #create the sim dict
    factors={x:round(scores[x]*weights[x],2) for x in scores}

    #sort factors by sim
    sorted_factors=[factor for factor in sorted(factors.items(), key=lambda x:x[1],reverse=True) if factor[1]>0]

    #return overall score and explanations
    return round(np.sum(list(factors.values())),2),sorted_factors

In [7]:

sim('Toy Story',
       'Toy Story',
       movies,
       {'ryear':1, 'genre':1, 'director':1, 'rating':1, 'star':1, 'overview': 1},
       overview_sim_matrix,
       title_index)

(np.float64(5.83),
 [('star', 1.0),
  ('director', 1.0),
  ('genre', 1.0),
  ('ryear', 1.0),
  ('overview', np.float32(1.0)),
  ('rating', 0.83)])

In [8]:
def recommend(input_title:str,
              k:int,
              movies:list,
              weights:dict,
              overview_sim_matrix:Tensor,
              title_index:dict
              )->list:
    """
    Generates top-k movie recommendations similar to the input title.

    Args:
        input_title (str): The title of the movie to find recommendations for.
        k (int): The number of recommendations to return.
        movies (list): List of movie objects.
        weights (dict): User preference weights for similarity factors.
        overview_sim_matrix (Tensor): Similarity matrix for overviews.
        title_index (dict): Index mapping titles to movie list positions.

    Returns:
        list: A list of top-k recommended items, where each item is a tuple
              (title, (score, explanation)).
    """

    results={} # recommendations

    for candidate in movies: # for each candidate

        #get the similarity and the explanation
        my_sim,my_exp=sim(candidate.title,input_title,movies, weights,overview_sim_matrix,title_index)

        #remember
        results[candidate.title]=(my_sim,my_exp)

    #store, slice, return
    return sorted(results.items(),key=lambda x:x[1][0],reverse=True)[:k]

In [9]:

weights={'ryear':1, 'genre':1, 'director':1, 'rating':1, 'star':1, 'overview': 1}

recommend('Toy Story',5,movies,weights,overview_sim_matrix,title_index)

[('Toy Story',
  (np.float64(5.83),
   [('star', 1.0),
    ('director', 1.0),
    ('genre', 1.0),
    ('ryear', 1.0),
    ('overview', np.float32(1.0)),
    ('rating', 0.83)])),
 ('Toy Story 2',
  (np.float64(4.45),
   [('director', 1.0),
    ('genre', 1.0),
    ('ryear', 0.96),
    ('rating', 0.79),
    ('overview', np.float32(0.37)),
    ('star', 0.33)])),
 ('Toy Story 3',
  (np.float64(3.35),
   [('genre', 1.0),
    ('ryear', 0.85),
    ('rating', 0.82),
    ('overview', np.float32(0.35)),
    ('star', 0.33)])),
 ('Toy Story 4',
  (np.float64(3.27),
   [('genre', 1.0),
    ('rating', 0.78),
    ('ryear', 0.76),
    ('overview', np.float32(0.4)),
    ('star', 0.33)])),
 ('Monsters, Inc.',
  (np.float64(2.99),
   [('genre', 1.0),
    ('ryear', 0.94),
    ('rating', 0.81),
    ('overview', np.float32(0.24))]))]

<h1>Simulation<h1>

In [10]:
@dataclass
class User:
    """
    Represents a simulated user in the recommendation system.
    """
    seed_movies:list # latent movies that the user likes
    likes:list # known movies that the user has liked
    dislikes:list # known movies that the user has disliked
    weights:dict # user preferences
    like_threshold:float # similarity threshold for liking a movie

In [11]:
from random import random,sample,shuffle

#create fake users
def generate_users(movies:list, # list of movie objects
             overview_sim_matrix, # movie-to-movie sim matrix
             title_index, # index that maps each title to a position in the movies list
             user_num:int=100,  # number of users to generaate
             seed_movie_num:int=5, # number of random seed movies to choose
             factors=['ryear','genre','director','rating','star','overview'], # factors to consider
             std_multiplier:float=1.5 # used to tune the 'Like' threshold
            ):
    """
    Generates a population of fake users with random preferences.

    For each user:
    1. Assigns random weights to similarity factors.
    2. Selects a set of random 'seed' movies they like.
    3. Calculates a 'like_threshold' based on the similarity of the seed movies
       to the rest of the catalog.

    Args:
        movies (list): List of movie objects.
        overview_sim_matrix (Tensor): Similarity matrix.
        title_index (dict): Title to index mapping.
        user_num (int): Number of users to generate.
        seed_movie_num (int): Number of seed movies per user.
        factors (list): List of factors to generate weights for.
        std_multiplier (float): Multiplier for standard deviation to set like threshold.

    Returns:
        list: A list of User objects.
    """

    users=[]# list of fake users

    for i in range(user_num):  # for each fake user to create
        print(i)

        weights={} # user preferences for each factor

        for factor in factors: # for each factor
            weights[factor]=round(random(),2) # sample a random preference value (weight)

        seed_movies=sample(movies,seed_movie_num)# sample 5 seed movies

        '''
        Compute the "like" threshold for this user
        If a movie has an above-threshold similarity with (at least)
        one of the seed movies, then we assume that the user will like it.
        The threshold is defined to be equal to the average sim of all movies
        with the seed movies, +1 stdev
        '''

        sim_scores=[]

        s=0
        for seed_movie in seed_movies: # for each seed movie of this user

            for candidate in movies: # for each other movie

                #compute seed-candidate sim
                val=sim(candidate.title,
                           seed_movie.title,
                           movies,
                           weights,
                           overview_sim_matrix,
                           title_index)[0]

                sim_scores.append(val) # store

        like_threshold=np.mean(sim_scores)+ std_multiplier*np.std(sim_scores) # threshold is mean + 1 std dev

        # remember the user
        users.append(User(seed_movies,
                          [], # liked movies
                          [], # disliked movies
                          weights, # preferences
                          round(like_threshold,2)) # like threshold
                    )


    return users

In [12]:
users=generate_users(movies,overview_sim_matrix,title_index)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


In [13]:
users[4].weights

{'ryear': 0.41,
 'genre': 0.81,
 'director': 0.84,
 'rating': 0.99,
 'star': 0.03,
 'overview': 0.47}

In [14]:
def random_recommender(
            users:list, # fake users
             movies:list, # movie list
            title_index:dict, # maps titles to indices in the movies list
             overview_sim_matrix:Tensor, # movie to movie sim matrix
            recnum_per_user:int=50, # number of recommendations to make
             factors=['ryear','genre','director','rating','star','overview'] # factors to consider
            ):
    """
    Simulates a recommendation process for a list of users.

    For each user, random movies are picked as recommendations.
    The user 'likes' the recommendation if it is similar enough (above threshold)
    to any of their seed movies.

    Args:
        users (list): List of User objects.
        movies (list): List of movie objects.
        title_index (dict): Title to index mapping.
        overview_sim_matrix (Tensor): Similarity matrix.
        recnum_per_user (int): Number of random recommendations to test per user.
        factors (list): Factors used.

    Returns:
        float: The average number of 'likes' per user out of the recommended movies.
    """


    ress=[]

    for user in users: # for each fake user

        #initialize likes and dislikes for this user
        user.likes=[]
        user.dislikes=[]

        recommended=set() # remember titles of recommended movies

        for i in range(recnum_per_user): # for each recommendation to make

            rec_movie=None # movie to be recommended

            # pick a random movie that has not been recommended before
            while rec_movie==None or rec_movie.title in recommended:
                rec_movie=sample(movies,1)[0]

            #remember the recommendation
            recommended.add(rec_movie.title)

            found_similar_seed=False # becomes true if the random movie is similar to a seed

            for sm in user.seed_movies: # for each seed movie for this user

                # compute the sim between the random movie and the seed movie
                val=sim(rec_movie.title,
                       sm.title,
                       movies,
                       user.weights,
                       overview_sim_matrix,
                       title_index)[0]

                # if the sim is over the like threshold of this user
                if val>user.like_threshold:
                    found_similar_seed=True
                    break

            if found_similar_seed: # similar seed found, the user will like this movie
                user.likes.append(rec_movie)
                #print('YES',rec_movie.title,len(user.likes))
            else:
                user.dislikes.append(rec_movie)
                #print('NO',rec_movie.title,len(user.dislikes))

        ress.append(len(user.likes))



        print('\n\nTotal Likes out of 50:', len(user.likes),'\n\n-----------------------\n')

    return np.mean(ress)

In [15]:
import numpy as np
random_recommender(users,movies,title_index, overview_sim_matrix)



Total Likes out of 50: 12 

-----------------------



Total Likes out of 50: 18 

-----------------------



Total Likes out of 50: 18 

-----------------------



Total Likes out of 50: 6 

-----------------------



Total Likes out of 50: 15 

-----------------------



Total Likes out of 50: 5 

-----------------------



Total Likes out of 50: 9 

-----------------------



Total Likes out of 50: 8 

-----------------------



Total Likes out of 50: 10 

-----------------------



Total Likes out of 50: 12 

-----------------------



Total Likes out of 50: 9 

-----------------------



Total Likes out of 50: 17 

-----------------------



Total Likes out of 50: 3 

-----------------------



Total Likes out of 50: 11 

-----------------------



Total Likes out of 50: 10 

-----------------------



Total Likes out of 50: 4 

-----------------------



Total Likes out of 50: 9 

-----------------------



Total Likes out of 50: 9 

-----------------------



Total Likes out o

np.float64(10.58)

In [16]:
# Smart recommender: instead of picking RANDOM movies,
# we use the recommend() function to find movies that are
# most similar to the user's seed movies.

def smart_recommender(
            users:list, # fake users
            movies:list, # movie list
            title_index:dict, # maps titles to indices
            overview_sim_matrix:Tensor, # movie to movie sim matrix
            recnum_per_user:int=50 # number of recommendations per user
            ):

    ress = []

    for user in users: # for each fake user

        user.likes = []
        user.dislikes = []
        recommended = set() # remember what we already recommended

        # for each seed movie, get the top-k most similar movies using the user's weights
        all_candidates = {} # title -> best similarity score

        for seed_movie in user.seed_movies: # for each of the user's seed movies

            # get the top recommendations for this seed movie using the user's own weights
            recs = recommend(seed_movie.title,
                           len(movies), # get ALL movies ranked by similarity
                           movies,
                           user.weights,
                           overview_sim_matrix,
                           title_index)

            for title, (score, explanation) in recs: # for each recommended movie
                # skip the seed movies themselves
                if title in [sm.title for sm in user.seed_movies]:
                    continue
                # keep the BEST score across all seed movies
                if title not in all_candidates or score > all_candidates[title]:
                    all_candidates[title] = score

        # sort all candidates by their best similarity score (highest first)
        sorted_candidates = sorted(all_candidates.items(), key=lambda x: x[1], reverse=True)

        # take the top recnum_per_user movies as our recommendations
        count = 0
        for title, score in sorted_candidates:
            if count >= recnum_per_user:
                break

            # now check if the user actually likes this movie (same logic as random_recommender)
            found_similar_seed = False
            for sm in user.seed_movies:
                val = sim(title,
                         sm.title,
                         movies,
                         user.weights,
                         overview_sim_matrix,
                         title_index)[0]

                if val > user.like_threshold:
                    found_similar_seed = True
                    break

            if found_similar_seed:
                user.likes.append(title)
            else:
                user.dislikes.append(title)

            count += 1

        ress.append(len(user.likes))

        print('\n\nTotal Likes out of 50:', len(user.likes), '\n\n-----------------------\n')

    return np.mean(ress)

In [17]:
# Run the smart recommender and compare with the random one
smart_avg = smart_recommender(users, movies, title_index, overview_sim_matrix)

print(f"\n\n{'='*50}")
print(f"Random Recommender average likes: 10.01 / 50")
print(f"Smart  Recommender average likes: {smart_avg:.2f} / 50")
print(f"{'='*50}")
print(f"Improvement: {smart_avg - 10.01:.2f} more likes per user on average")



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 24 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Likes out of 50: 50 

-----------------------



Total Li